In [1]:
import numpy as np
import pandas as pd
import joblib
import pickle
import time
import os
import psutil

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [2]:
X_train = np.load("../dataset/processed/X_train.npy")
X_test = np.load("../dataset/processed/X_test.npy")

y_train = np.load("../dataset/processed/y_train.npy")
y_test = np.load("../dataset/processed/y_test.npy")

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (65865, 10)
Testing : (16467, 10)


In [3]:
models = {
    "Random Forest": RandomForestClassifier(random_state=42),

    "AdaBoost": AdaBoostClassifier(random_state=42),

    "Gradient Boosting": GradientBoostingClassifier(random_state=42),

    "KNN": KNeighborsClassifier(),

    "Naive Bayes": GaussianNB()
}

In [4]:
results = []

In [5]:
for name, model in models.items():

    print("="*60)
    print(f"Training {name}")

    process = psutil.Process(os.getpid())

    cpu_before = psutil.cpu_percent(interval=None)
    ram_before = process.memory_info().rss

    start_train = time.time()

    model.fit(X_train, y_train)

    end_train = time.time()

    cpu_after = psutil.cpu_percent(interval=None)
    ram_after = process.memory_info().rss

    start_predict = time.time()

    predictions = model.predict(X_test)

    end_predict = time.time()

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(y_test, predictions)

    recall = recall_score(y_test, predictions)

    f1 = f1_score(y_test, predictions)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions
    ).ravel()

    specificity = tn / (tn + fp)

    fpr = fp / (fp + tn)

    roc = roc_auc_score(
        y_test,
        predictions
    )

    filename = "../models/" + name.replace(" ", "_") + ".pkl"

    joblib.dump(model, filename)

    model_size = os.path.getsize(filename) / (1024 * 1024)

    results.append({

        "Model": name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "Specificity": specificity,

        "FPR": fpr,

        "ROC_AUC": roc,

        "CPU_Usage": cpu_after,

        "RAM_Usage_MB":
            (ram_after - ram_before)/(1024*1024),

        "Training_Time":
            end_train-start_train,

        "Prediction_Time":
            end_predict-start_predict,

        "Model_Size_MB":
            model_size

    })

    print("Finished")

Training Random Forest
Finished
Training AdaBoost
Finished
Training Gradient Boosting
Finished
Training KNN
Finished
Training Naive Bayes
Finished


In [6]:
results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1,Specificity,FPR,ROC_AUC,CPU_Usage,RAM_Usage_MB,Training_Time,Prediction_Time,Model_Size_MB
0,Random Forest,0.932228,0.949972,0.925664,0.937661,0.940270,0.059730,0.932967,57.7,33.457031,8.589992,0.334809,52.544717
1,AdaBoost,0.851764,0.941498,0.779199,0.852694,0.940676,0.059324,0.859937,76.7,-8.664062,3.307499,0.210770,0.031834
2,Gradient Boosting,0.920447,0.959157,0.893570,0.925203,0.953378,0.046622,0.923474,59.1,-7.199219,8.918243,0.039536,0.134132
3,KNN,0.907451,0.934755,0.894342,0.914102,0.923514,0.076486,0.908928,87.1,5.417969,0.283321,1.274288,11.806433
4,Naive Bayes,0.511751,0.763468,0.164112,0.270153,0.937703,0.062297,0.550907,62.5,0.046875,0.025984,0.004064,0.001044


In [8]:
results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1,Specificity,FPR,ROC_AUC,CPU_Usage,RAM_Usage_MB,Training_Time,Prediction_Time,Model_Size_MB
0,Random Forest,0.932228,0.949972,0.925664,0.937661,0.940270,0.059730,0.932967,57.7,33.457031,8.589992,0.334809,52.544717
2,Gradient Boosting,0.920447,0.959157,0.893570,0.925203,0.953378,0.046622,0.923474,59.1,-7.199219,8.918243,0.039536,0.134132
3,KNN,0.907451,0.934755,0.894342,0.914102,0.923514,0.076486,0.908928,87.1,5.417969,0.283321,1.274288,11.806433
1,AdaBoost,0.851764,0.941498,0.779199,0.852694,0.940676,0.059324,0.859937,76.7,-8.664062,3.307499,0.210770,0.031834
4,Naive Bayes,0.511751,0.763468,0.164112,0.270153,0.937703,0.062297,0.550907,62.5,0.046875,0.025984,0.004064,0.001044


In [9]:
results_df.to_csv(
    "../results/base_model_metrics.csv",
    index=False
)

print("Results saved.")

Results saved.


In [10]:
results_df.to_excel(
    "../results/base_model_metrics.xlsx",
    index=False
)

print("Excel saved.")

Excel saved.


In [11]:
best = results_df.iloc[0]

print("="*60)

print("BEST BASE MODEL")

print("="*60)

print(best)

BEST BASE MODEL
Model              Random Forest
Accuracy                0.932228
Precision               0.949972
Recall                  0.925664
F1                      0.937661
Specificity              0.94027
FPR                      0.05973
ROC_AUC                 0.932967
CPU_Usage                   57.7
RAM_Usage_MB           33.457031
Training_Time           8.589992
Prediction_Time         0.334809
Model_Size_MB          52.544717
Name: 0, dtype: object
